In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector


def build_backbone_qnn(n_features, reps=1, entanglement="ring"):
    n_qubits = n_features
    x_params = ParameterVector("x", n_features)
    theta_params = ParameterVector("θ", length=2 * n_qubits * reps)

    qc = QuantumCircuit(n_qubits)

    # Feature encoding (same idea as before; keep simple)
    for i in range(n_qubits):
        qc.ry(x_params[i], i)

    t = 0
    for _ in range(reps):
        for q in range(n_qubits):
            qc.ry(theta_params[t], q); t += 1
            qc.rz(theta_params[t], q); t += 1

        if entanglement == "ring":
            for q in range(n_qubits - 1):
                qc.cx(q, q + 1)
            qc.cx(n_qubits - 1, 0)
        elif entanglement == "all":
            for i in range(n_qubits):
                for j in range(i + 1, n_qubits):
                    qc.cx(i, j)
        else:
            raise ValueError("entanglement must be 'ring' or 'all'")

    # IMPORTANT: use single observable (works in your environment)
    observable = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])
    estimator = Estimator()

    qnn = EstimatorQNN(
        circuit=qc,
        estimator=estimator,
        observables=observable,
        input_params=list(x_params),
        weight_params=list(theta_params),
    )
    return qnn

In [3]:
class BinaryFireModel(nn.Module):
    """
    Outputs a logit (unbounded real). Use sigmoid(logit) for probability.
    """
    def __init__(self, qnn):
        super().__init__()
        self.q = TorchConnector(qnn)      # output ~[-1,1]
        self.head = nn.Linear(1, 1)       # logit = a*q + b

    def forward(self, X):
        return self.head(self.q(X))       # logits, shape (batch, 1)


def train_binary(model, X_train_a, y_train_raw, batch_size=256, epochs=10, lr=0.01):
    X = torch.tensor(X_train_a, dtype=torch.float32)
    y_bin = (np.asarray(y_train_raw) > 0).astype(np.float32)
    y = torch.tensor(y_bin, dtype=torch.float32).view(-1, 1)

    # pos_weight = (#neg / #pos) helps with imbalance
    n_pos = float(y_bin.sum())
    n_neg = float(len(y_bin) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], dtype=torch.float32)

    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        n = 0
        for xb, yb in loader:
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            total += loss.item() * xb.shape[0]
            n += xb.shape[0]
        print(f"[binary] epoch={epoch+1:03d} loss={total/n:.6f}  pos_weight={pos_weight.item():.2f}")

    return model

In [4]:
class PositiveCountModel(nn.Module):
    """
    Predicts mean count conditional on y>0, enforced positive via softplus.
    """
    def __init__(self, qnn):
        super().__init__()
        self.q = TorchConnector(qnn)      # output ~[-1,1]
        self.head = nn.Linear(1, 1)
        self.softplus = nn.Softplus()

    def forward(self, X):
        return self.softplus(self.head(self.q(X)))  # mu(x) >= 0


def train_positive_regressor(model, X_train_a, y_train_raw, batch_size=256, epochs=10, lr=0.01):
    y = np.asarray(y_train_raw).astype(np.float32)
    pos_idx = np.where(y > 0)[0]

    if len(pos_idx) < 5:
        raise ValueError(f"Too few positive samples to train regressor: {len(pos_idx)}")

    Xpos = torch.tensor(X_train_a[pos_idx], dtype=torch.float32)
    ypos = torch.tensor(y[pos_idx], dtype=torch.float32).view(-1, 1)

    loss_fn = nn.PoissonNLLLoss(log_input=False, full=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loader = DataLoader(TensorDataset(Xpos, ypos), batch_size=min(batch_size, len(pos_idx)), shuffle=True)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        n = 0
        for xb, yb in loader:
            opt.zero_grad()
            mu = model(xb)
            loss = loss_fn(mu, yb)
            loss.backward()
            opt.step()
            total += loss.item() * xb.shape[0]
            n += xb.shape[0]
        print(f"[pos-reg] epoch={epoch+1:03d} loss={total/n:.6f}  n_pos={len(pos_idx)}")

    return model

In [5]:
def predict_expected_fires(binary_model, pos_model, X_a, batch_size=512):
    """
    y_hat = sigmoid(logit(x)) * mu_pos(x)
    """
    binary_model.eval()
    pos_model.eval()

    X = torch.tensor(X_a, dtype=torch.float32)
    n = X.shape[0]

    out = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            xb = X[start:start + batch_size]
            logits = binary_model(xb)
            p = torch.sigmoid(logits)          # (batch,1)
            mu = pos_model(xb)                 # (batch,1)
            yhat = (p * mu).cpu().numpy()
            out.append(yhat)

    return np.vstack(out).ravel()

In [6]:
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score

def evaluate_two_stage(binary_model, pos_model, X_test_a, y_test_raw, batch_size=512):
    y_test_raw = np.asarray(y_test_raw).astype(float)

    # expected fires prediction
    y_pred = predict_expected_fires(binary_model, pos_model, X_test_a, batch_size=batch_size)

    mse = mean_squared_error(y_test_raw, y_pred)
    r2 = r2_score(y_test_raw, y_pred)

    # AUC: use binary model score (probability) as the ranking score
    y_true_bin = (y_test_raw > 0).astype(int)

    # probability score
    binary_model.eval()
    X = torch.tensor(X_test_a, dtype=torch.float32)
    probs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = X[start:start+batch_size]
            probs.append(torch.sigmoid(binary_model(xb)).cpu().numpy())
    p_score = np.vstack(probs).ravel()

    auc = roc_auc_score(y_true_bin, p_score) if len(np.unique(y_true_bin)) == 2 else np.nan

    return {"mse": mse, "r2": r2, "auc_roc": auc}, y_pred, p_score

In [7]:
def make_balanced_subset(X, y, fire_threshold=0.0, pos_fraction=0.30, n_samples=2000, seed=0):
    """
    Returns a subsample of (X, y) with a higher proportion of "fire" rows.

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
    y : np.ndarray, shape (N,)
    fire_threshold : float
        Define fire as y > fire_threshold.
    pos_fraction : float in (0,1)
        Desired fraction of positives in the returned subset (e.g., 0.3 => 30% fires).
    n_samples : int
        Total number of samples to return.
    seed : int
        RNG seed.

    Notes
    -----
    - This oversamples positives if you ask for more positives than exist.
    - Negatives are sampled without replacement (if enough exist).
    """
    rng = np.random.default_rng(seed)

    y = np.asarray(y)
    pos_idx = np.where(y > fire_threshold)[0]
    neg_idx = np.where(y <= fire_threshold)[0]

    n_pos = int(round(n_samples * pos_fraction))
    n_neg = n_samples - n_pos

    if len(pos_idx) == 0:
        raise ValueError("No positive (fire) examples found for the given threshold.")
    if len(neg_idx) == 0:
        raise ValueError("No negative examples found for the given threshold.")

    # Sample positives (with replacement if needed)
    replace_pos = n_pos > len(pos_idx)
    pos_sel = rng.choice(pos_idx, size=n_pos, replace=replace_pos)

    # Sample negatives (without replacement if possible; otherwise with replacement)
    replace_neg = n_neg > len(neg_idx)
    neg_sel = rng.choice(neg_idx, size=n_neg, replace=replace_neg)

    idx = np.concatenate([pos_sel, neg_sel])
    rng.shuffle(idx)

    return X[idx], y[idx], idx


# ---- Example insertion point ----
# Use this right before you create torch tensors / DataLoader for training:
# (Choose pos_fraction and n_samples as you like)


# Then train on x_train_bal, y_train_bal instead of x_train_a, y_train

In [8]:
def sample_rebalanced_from_full(X, y, fire_threshold=0.0, pos_fraction=0.30,
                                n_samples=20000, seed=0,
                                oversample_pos_if_needed=True):
    """
    Draw a new dataset from the full pool (X,y) with a higher positive (fire) fraction.

    - Samples negatives without replacement (if possible).
    - Samples positives without replacement if enough positives exist, otherwise with replacement
      if oversample_pos_if_needed=True.
    """
    rng = np.random.default_rng(seed)
    y = np.asarray(y)

    pos_idx = np.where(y > fire_threshold)[0]
    neg_idx = np.where(y <= fire_threshold)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        raise ValueError("Need at least one positive and one negative example.")

    n_pos = int(round(n_samples * pos_fraction))
    n_neg = n_samples - n_pos

    # positives
    if n_pos <= len(pos_idx):
        pos_sel = rng.choice(pos_idx, size=n_pos, replace=False)
    else:
        if not oversample_pos_if_needed:
            raise ValueError(f"Requested {n_pos} positives but only {len(pos_idx)} exist.")
        pos_sel = rng.choice(pos_idx, size=n_pos, replace=True)  # duplicates only if needed

    # negatives
    neg_sel = rng.choice(neg_idx, size=n_neg, replace=(n_neg > len(neg_idx)))

    idx = np.concatenate([pos_sel, neg_sel])
    rng.shuffle(idx)

    return X[idx], y[idx], idx

In [9]:
import pandas as pd

df = pd.read_csv("features.csv")
dfx = df[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]
x_a = dfx.values.astype(float)
y_a = df["target"].values.astype(float)

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

n_samples = 2000

def make_train_test_with_fire_fraction(
    X_all,
    y_all,
    n_train=2000,
    n_test=2000,
    fire_threshold=0.0,
    train_fire_fraction=0.30,
    test_fire_fraction=None,          # None -> keep natural test distribution
    seed=0,
):
    """
    Build new (train,test) splits sampled from the FULL dataset with a desired fire fraction.
    Then scale features exactly as you did:
      StandardScaler -> MinMaxScaler(feature_range=(0,2pi))

    Returns:
      x_train_a, x_test_a, y_train, y_test, plus scalers and indices used.
    """

    rng = np.random.default_rng(seed)
    X_all = np.asarray(X_all)
    y_all = np.asarray(y_all)

    pos_idx = np.where(y_all > fire_threshold)[0]
    neg_idx = np.where(y_all <= fire_threshold)[0]
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        raise ValueError("Need at least one fire and one non-fire sample in the full dataset.")

    def sample_indices(n, fire_frac):
        """Sample n indices with fire fraction fire_frac. Uses replacement only if necessary."""
        n_pos = int(round(n * fire_frac))
        n_neg = n - n_pos

        if n_pos > len(pos_idx):
            # Only way is to oversample positives (duplicate rows).
            pos_sel = rng.choice(pos_idx, size=n_pos, replace=True)
        else:
            pos_sel = rng.choice(pos_idx, size=n_pos, replace=False)

        if n_neg > len(neg_idx):
            neg_sel = rng.choice(neg_idx, size=n_neg, replace=True)
        else:
            neg_sel = rng.choice(neg_idx, size=n_neg, replace=False)

        idx = np.concatenate([pos_sel, neg_sel])
        rng.shuffle(idx)
        return idx

    # --- Train sample (re-fractioned) ---
    train_idx = sample_indices(n_train, train_fire_fraction)

    # --- Test sample ---
    if test_fire_fraction is None:
        # "Natural" test distribution (random sample from all data, no fraction constraint)
        # Keep it disjoint from train where possible:
        remaining = np.setdiff1d(np.arange(len(y_all)), train_idx, assume_unique=False)
        if len(remaining) >= n_test:
            test_idx = rng.choice(remaining, size=n_test, replace=False)
        else:
            # fall back: allow overlap only if dataset is too small (shouldn't happen for 155k)
            test_idx = rng.choice(np.arange(len(y_all)), size=n_test, replace=False)
    else:
        # Re-fractioned test distribution too
        test_idx = sample_indices(n_test, test_fire_fraction)

        # Try to enforce disjointness from train by resampling a few times if overlap exists
        # (disjoint is strongly preferred to avoid leakage)
        for _ in range(20):
            overlap = np.intersect1d(train_idx, test_idx)
            if len(overlap) == 0:
                break
            test_idx = sample_indices(n_test, test_fire_fraction)

    # Build raw splits
    x_train = X_all[train_idx]
    y_train = y_all[train_idx]
    x_test  = X_all[test_idx]
    y_test  = y_all[test_idx]

    # --- Scaling: StandardScaler -> MinMaxScaler to [0, 2π] ---
    x_scaler = StandardScaler()
    x_train_s = x_scaler.fit_transform(x_train)
    x_test_s  = x_scaler.transform(x_test)

    angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
    x_train_a = angle_scaler.fit_transform(x_train_s)
    x_test_a  = angle_scaler.transform(x_test_s)

    info = {
        "train_idx": train_idx,
        "test_idx": test_idx,
        "train_fire_fraction": float(np.mean(y_train > fire_threshold)),
        "test_fire_fraction": float(np.mean(y_test > fire_threshold)),
        "n_train_pos": int(np.sum(y_train > fire_threshold)),
        "n_test_pos": int(np.sum(y_test > fire_threshold)),
        "x_scaler": x_scaler,
        "angle_scaler": angle_scaler,
    }

    return x_train_a, x_test_a, y_train, y_test, info

x_train_a, x_test_a, y_train, y_test, info = make_train_test_with_fire_fraction(
    X_all=x_a,
    y_all=y_a,
    n_train=2000,
    n_test=2000,
    fire_threshold=0.0,
    train_fire_fraction=0.30,
    test_fire_fraction=0.30,   # or None to keep natural test distribution
    seed=0
)

print(info["train_fire_fraction"], info["n_train_pos"])
print(info["test_fire_fraction"], info["n_test_pos"])

0.3 600
0.3 600


In [10]:
n_features = x_train_a.shape[1]  # you said 9
reps = 1

# Build separate QNNs (simplest; avoids entangling the objectives)
qnn_bin = build_backbone_qnn(n_features=n_features, reps=reps, entanglement="ring")
qnn_pos = build_backbone_qnn(n_features=n_features, reps=reps, entanglement="ring")

binary_model = BinaryFireModel(qnn_bin)
pos_model = PositiveCountModel(qnn_pos)

# Train
binary_model = train_binary(binary_model, x_train_a, y_train, batch_size=256, epochs=10, lr=0.01)
pos_model = train_positive_regressor(pos_model, x_train_a, y_train, batch_size=64, epochs=5, lr=0.01)

# Evaluate
metrics, y_pred, p_score = evaluate_two_stage(binary_model, pos_model, x_test_a, y_test, batch_size=512)
print(metrics)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.
No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


[binary] epoch=001 loss=1.082460  pos_weight=2.33
[binary] epoch=002 loss=1.061811  pos_weight=2.33
[binary] epoch=003 loss=1.043836  pos_weight=2.33
[binary] epoch=004 loss=1.028741  pos_weight=2.33
[binary] epoch=005 loss=1.015306  pos_weight=2.33
[binary] epoch=006 loss=1.004484  pos_weight=2.33
[binary] epoch=007 loss=0.995206  pos_weight=2.33
[binary] epoch=008 loss=0.988960  pos_weight=2.33
[binary] epoch=009 loss=0.983370  pos_weight=2.33
[binary] epoch=010 loss=0.979555  pos_weight=2.33
[pos-reg] epoch=001 loss=1.147415  n_pos=600
[pos-reg] epoch=002 loss=1.115118  n_pos=600
[pos-reg] epoch=003 loss=1.086467  n_pos=600
[pos-reg] epoch=004 loss=1.062063  n_pos=600
[pos-reg] epoch=005 loss=1.041482  n_pos=600
{'mse': 0.12868021142053457, 'r2': -0.02371244824421348, 'auc_roc': 0.5149851190476191}


In [12]:
import numpy as np
import torch

def predict_expected_fires(model, X_all_a, batch_size=1024):
    """
    Returns expected-fire-count predictions for every row in X_all_a.

    model should output raw fire counts (e.g., includes Linear + Softplus head).
    """
    model.eval()

    X = torch.tensor(X_all_a, dtype=torch.float32)
    n = X.shape[0]

    preds = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            xb = X[start:start + batch_size]
            yb = model(xb)  # (batch, 1)
            preds.append(yb.detach().cpu().numpy())

    y_pred_all = np.vstack(preds).ravel()
    return y_pred_all

df = pd.read_csv("features.csv")
dfx = df[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]
x_a = dfx.values.astype(float)
#y_a = df_full["target"].values.astype(float)

y_expected_all = predict_expected_fires(pos_model, x_a, batch_size=1024)


print(y_expected_all.shape)
print("first 10 predictions:", y_expected_all[:1000])
print(min(y_expected_all))

(155580,)
first 10 predictions: [0.52073365 0.5152106  0.516296   0.5156901  0.51697403 0.51884186
 0.5226533  0.5295309  0.5313594  0.52754104 0.52717584 0.5188876
 0.5231081  0.5188957  0.5173589  0.5201149  0.51751006 0.52416825
 0.52406037 0.52982956 0.54406327 0.5487004  0.5438496  0.531848
 0.5262227  0.5258961  0.5234184  0.51972246 0.52481115 0.5274075
 0.53068334 0.54272956 0.55222887 0.5486477  0.5509929  0.53807414
 0.53049386 0.52819335 0.52630115 0.52522683 0.5239586  0.5292273
 0.5323749  0.54255396 0.5497169  0.5552782  0.55042243 0.5445788
 0.53920114 0.53047216 0.5241428  0.51808596 0.5239719  0.5264333
 0.5340688  0.54568845 0.54977375 0.556724   0.5536607  0.5368467
 0.52250755 0.52242196 0.52047676 0.5185219  0.515776   0.51611644
 0.5189745  0.52284443 0.52452725 0.5331785  0.5293592  0.5254945
 0.5213984  0.51696587 0.51966697 0.51668876 0.5171766  0.51887184
 0.52243024 0.5323954  0.5359765  0.5459417  0.54038143 0.53563076
 0.5276969  0.5224278  0.52558905 0.516

In [13]:
# prints ALL trainable parameters (names + full tensors)

for name, p in binary_model.named_parameters():
    if p.requires_grad:
        print(name)
        print(p.detach().cpu())
        print()

for name, p in pos_model.named_parameters():
    if p.requires_grad:
        print(name)
        print(p.detach().cpu())
        print()

q.weight
tensor([-0.2582, -0.0138, -0.4097, -0.6467, -0.2800, -0.0862, -0.8157,  0.8303,
        -1.0863, -0.1807,  0.3949, -0.2046,  0.0917,  0.2399,  0.4218, -0.0446,
         0.0271, -0.3926])

head.weight
tensor([[-0.1844]])

head.bias
tensor([0.2021])

q.weight
tensor([ 0.1352,  0.7468, -0.9394,  0.1928, -1.0620,  0.3568, -0.7580,  0.4103,
        -0.8385, -0.4812, -0.3342, -0.2672, -0.8299, -0.9860, -0.6483, -0.3164,
         0.4887, -0.3726])

head.weight
tensor([[0.3861]])

head.bias
tensor([-0.4118])



In [23]:
import numpy as np
import pandas as pd
import torch

def export_location_timeseries_csv_two_stage(
    binary_model,
    pos_model,
    df,
    x_scaler,
    angle_scaler,
    lat_value,
    lon_value,
    tol=1e-6,
    year_range=None,         # optional (min_year, max_year)
    row_range=None,          # optional slice AFTER sorting
    batch_size=1024,
    out_csv_path="lat_lon_timeseries.csv",
):
    # exact columns from your CSV
    feature_cols = [
        "month_sin","month_cos","year","avg_tmax_c","temp_range",
        "tot_prcp_mm","dryness_3m_avg","latitude","longitude"
    ]
    date_col = "year_month"

    # sanity
    missing = [c for c in feature_cols + [date_col] if c not in df.columns]
    if missing:
        raise ValueError(f"df missing columns: {missing}")

    X_all = df[feature_cols].to_numpy(dtype=float)         # (N,9)
    ym_all = df[date_col].astype(str).to_numpy()           # (N,)

    # filter by lat/lon (feature indices 7 and 8)
    lat = X_all[:, 7]
    lon = X_all[:, 8]
    mask = (np.abs(lat - lat_value) <= tol) & (np.abs(lon - lon_value) <= tol)

    idx = np.where(mask)[0]
    if len(idx) == 0:
        raise ValueError("No rows matched the given (lat, lon).")

    Xf = X_all[idx]
    ym = ym_all[idx]

    # parse YYYY-MM
    years = np.array([int(s[:4]) for s in ym], dtype=int)
    months = np.array([int(s[5:7]) for s in ym], dtype=int)

    if year_range is not None:
        y0, y1 = year_range
        keep = (years >= y0) & (years <= y1)
        Xf = Xf[keep]
        years = years[keep]
        months = months[keep]
        if len(Xf) == 0:
            raise ValueError("year_range produced an empty selection.")

    # sort by time
    order = np.lexsort((months, years))
    Xf = Xf[order]
    years = years[order]
    months = months[order]

    if row_range is not None:
        a, b = row_range
        Xf = Xf[a:b]
        years = years[a:b]
        months = months[a:b]
        if len(Xf) == 0:
            raise ValueError("row_range produced an empty selection.")

    # month index from start (no rollover)
    month_from_start = ((years - years[0]) * 12 + (months - months[0])).astype(int)

    if len(np.unique(month_from_start)) != len(month_from_start):
        raise ValueError("Duplicate month_from_start detected (duplicate year_month rows).")

    # scale and predict
    Xa = angle_scaler.transform(x_scaler.transform(Xf))

    binary_model.eval()
    pos_model.eval()
    Xt = torch.tensor(Xa, dtype=torch.float32)

    preds = []
    with torch.no_grad():
        for start in range(0, len(Xt), batch_size):
            xb = Xt[start:start + batch_size]
            p = torch.sigmoid(binary_model(xb))
            mu = pos_model(xb)
            preds.append((p * mu).cpu().numpy().ravel())
    y_pred = np.concatenate(preds)

    out = pd.DataFrame({"month_from_start": month_from_start, "prediction": y_pred.astype(float)})
    out.to_csv(out_csv_path, index=False)
    return out

df_raw = pd.read_csv("features.csv")

out_df = export_location_timeseries_csv_two_stage(
    binary_model=binary_model,
    pos_model=pos_model,
    df=df_raw,
    x_scaler=x_scaler,
    angle_scaler=angle_scaler,
    lat_value=0.15040841135447547,
    lon_value=0.5939119939852004,
    tol=1e-6,
    out_csv_path="lat_lon_timeseries_2.csv",
)